In [0]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    roc_auc_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F
import mlflow
import mlflow.sklearn

In [0]:
# Read train/test tables (already have features vector + label from MLlib pipeline)
train_spark = spark.read.table("teams.sensorx.train_data")
test_spark = spark.read.table("teams.sensorx.test_data")

# Sample both classes equally to preserve natural distribution (~78% safe / 22% fault)
# 10% gives ~500K train rows — fits in 16GB driver while reflecting production ratio
SAMPLE_FRAC = 0.1
train_sampled = train_spark.sample(fraction=SAMPLE_FRAC, seed=42)
test_sampled = test_spark.sample(fraction=SAMPLE_FRAC, seed=42)

print(f"Train sampled: {train_sampled.count():,} rows")
print(f"Test sampled:  {test_sampled.count():,} rows")
train_sampled.groupBy("label").count().orderBy("label").show()
test_sampled.groupBy("label").count().orderBy("label").show()

# Write expanded features to parquet (bypasses spark.driver.maxResultSize limit)
tmp_train = "/Volumes/teams/sensorx/data-dump/_temp_train_expanded"
tmp_test = "/Volumes/teams/sensorx/data-dump/_temp_test_expanded"

(train_sampled
    .select(vector_to_array("features").alias("features"), "label")
    .write.mode("overwrite").parquet(tmp_train))

(test_sampled
    .select(vector_to_array("features").alias("features"), "label")
    .write.mode("overwrite").parquet(tmp_test))

# Read directly with pandas (no Spark collect needed)
train_pdf = pd.read_parquet(tmp_train)
test_pdf = pd.read_parquet(tmp_test)

# Expand feature arrays into numpy (float32 to halve memory)
X_train = np.vstack(train_pdf["features"].values).astype(np.float32)
y_train = train_pdf["label"].values

X_test = np.vstack(test_pdf["features"].values).astype(np.float32)
y_test = test_pdf["label"].values

# Free pandas intermediates
del train_pdf, test_pdf

print(f"\nTrain: {X_train.shape[0]:,} rows, {X_train.shape[1]} features")
print(f"Test:  {X_test.shape[0]:,} rows, {X_test.shape[1]} features")
print(f"Train label dist: {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"Test label dist:  {dict(zip(*np.unique(y_test, return_counts=True)))}")

In [0]:
# Scale features (fit on train only to avoid data leakage)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Feature means (first 6): {X_train_scaled.mean(axis=0)[:6].round(4)}")
print(f"Feature stds  (first 6): {X_train_scaled.std(axis=0)[:6].round(4)}")

In [0]:
# Train MLP — same architecture as MLlib version: 112 -> 16 -> 16 -> 2
# sklearn infers input (112) and output (2) layers from data shape automatically
mlp = MLPClassifier(
    hidden_layer_sizes=(16, 16),
    activation="relu",
    solver="adam",
    learning_rate_init=0.001,
    max_iter=50,
    batch_size=128,
    random_state=42,
    verbose=True,
)

mlp.fit(X_train_scaled, y_train)
print(f"\nConverged: {mlp.n_iter_} iterations, final loss: {mlp.loss_:.6f}")

In [0]:
# Evaluate on test set
y_pred = mlp.predict(X_test_scaled)
y_prob = mlp.predict_proba(X_test_scaled)[:, 1]

# Training AUC
y_train_prob = mlp.predict_proba(X_train_scaled)[:, 1]
train_auc = roc_auc_score(y_train, y_train_prob)

auc = roc_auc_score(y_test, y_prob)
recall_0 = recall_score(y_test, y_pred, pos_label=0)
recall_1 = recall_score(y_test, y_pred, pos_label=1)
f1_0 = f1_score(y_test, y_pred, pos_label=0)
f1_1 = f1_score(y_test, y_pred, pos_label=1)

print(f"Train AUC: {train_auc:.4f}")
print(f"Test AUC:  {auc:.4f}")
print(f"Recall 0: {recall_0:.4f}  |  Recall 1: {recall_1:.4f}")
print(f"F1     0: {f1_0:.4f}  |  F1     1: {f1_1:.4f}")
print("\n" + classification_report(y_test, y_pred, target_names=["No fault", "Fault"]))

In [0]:
from sklearn.utils import resample
import time

# Balance training data: undersample safe class (class 0) to 2:1 ratio
# Natural distribution: ~78% class 0 (safe) vs ~22% class 1 (fault)
mask_0 = y_train == 0
mask_1 = y_train == 1
n_minority = mask_1.sum()  # faults are the minority
target_majority = 2 * n_minority  # 2:1 ratio (safe:fault)

# Undersample class 0
idx_0 = np.where(mask_0)[0]
idx_1 = np.where(mask_1)[0]
idx_0_down = resample(idx_0, replace=False, n_samples=target_majority, random_state=42)
idx_balanced = np.concatenate([idx_0_down, idx_1])
np.random.shuffle(idx_balanced)

X_train_bal = X_train_scaled[idx_balanced]
y_train_bal = y_train[idx_balanced]
print(f"Balanced train: {len(y_train_bal):,} rows")
print(f"  Class 0 (safe):  {(y_train_bal == 0).sum():,}")
print(f"  Class 1 (fault): {(y_train_bal == 1).sum():,}")

# --- Configurations to try ---
configs = {
    "baseline": dict(
        hidden_layer_sizes=(16, 16), activation="relu", solver="adam",
        learning_rate_init=0.001, max_iter=50, alpha=0.0001,
        early_stopping=False, batch_size=128, random_state=42,
    ),
    "balanced_2:1": dict(
        hidden_layer_sizes=(16, 16), activation="relu", solver="adam",
        learning_rate_init=0.001, max_iter=100, alpha=0.0001,
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=10,
        batch_size=128, random_state=42,
    ),
    "wider_net": dict(
        hidden_layer_sizes=(64, 32), activation="relu", solver="adam",
        learning_rate_init=0.001, max_iter=100, alpha=0.001,
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=10,
        batch_size=256, random_state=42,
    ),
    "deeper_net": dict(
        hidden_layer_sizes=(64, 32, 16), activation="relu", solver="adam",
        learning_rate_init=0.0005, max_iter=150, alpha=0.001,
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=15,
        batch_size=256, random_state=42,
    ),
    "tanh_regularized": dict(
        hidden_layer_sizes=(64, 32), activation="tanh", solver="adam",
        learning_rate_init=0.001, max_iter=100, alpha=0.01,
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=10,
        batch_size=256, random_state=42,
    ),
    "adaptive_lr": dict(
        hidden_layer_sizes=(64, 32), activation="relu", solver="adam",
        learning_rate="adaptive", learning_rate_init=0.001,
        max_iter=150, alpha=0.001,
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=10,
        batch_size=256, random_state=42,
    ),
}

results = []
for name, params in configs.items():
    print(f"\n{'='*60}")
    print(f"Training: {name}")
    print(f"  Params: layers={params['hidden_layer_sizes']}, alpha={params['alpha']}, act={params['activation']}")
    
    clf = MLPClassifier(**params, verbose=False)
    
    # Use balanced data for all configs except baseline
    X_tr = X_train_scaled if name == "baseline" else X_train_bal
    y_tr = y_train if name == "baseline" else y_train_bal
    
    t0 = time.time()
    clf.fit(X_tr, y_tr)
    elapsed = time.time() - t0
    
    # Evaluate on full (natural distribution) test set
    y_prob_tr = clf.predict_proba(X_train_scaled)[:, 1]
    y_prob_te = clf.predict_proba(X_test_scaled)[:, 1]
    y_pred_te = clf.predict(X_test_scaled)
    
    tr_auc = roc_auc_score(y_train, y_prob_tr)
    te_auc = roc_auc_score(y_test, y_prob_te)
    rec_0 = recall_score(y_test, y_pred_te, pos_label=0)
    rec_1 = recall_score(y_test, y_pred_te, pos_label=1)
    f1_fault = f1_score(y_test, y_pred_te, pos_label=1)
    
    results.append({
        "config": name,
        "layers": str(params["hidden_layer_sizes"]),
        "alpha": params["alpha"],
        "activation": params["activation"],
        "train_auc": tr_auc,
        "test_auc": te_auc,
        "gap": tr_auc - te_auc,
        "recall_0": rec_0,
        "recall_1": rec_1,
        "f1_fault": f1_fault,
        "iters": clf.n_iter_,
        "time_s": elapsed,
    })
    print(f"  Train AUC: {tr_auc:.4f} | Test AUC: {te_auc:.4f} | Gap: {tr_auc - te_auc:.4f}")
    print(f"  Recall 0: {rec_0:.4f} | Recall 1: {rec_1:.4f} | F1 fault: {f1_fault:.4f}")
    print(f"  Iters: {clf.n_iter_} | Time: {elapsed:.1f}s")

# Summary table
results_df = pd.DataFrame(results).sort_values("test_auc", ascending=False)
print(f"\n\n{'='*80}")
print("SUMMARY (sorted by Test AUC)")
print(f"{'='*80}")
display(results_df)